In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel, ViTModel, ViTImageProcessor

import numpy as np
from PIL import Image
from tqdm import tqdm

In [ ]:
dataset = load_dataset("Zoe3324/flickr30k-pairs")

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print(train_data[0])

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

In [4]:
def collate_fn(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]

    text_enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )

    image_enc = image_processor(
        images=images,
        return_tensors="pt"
    )

    return {
        "input_ids": text_enc["input_ids"],
        "attention_mask": text_enc["attention_mask"],
        "pixel_values": image_enc["pixel_values"],
    }

In [5]:
class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, q, k, v):
        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)

        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)
        return out

In [6]:
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        dim = 768

        self.cross_attn = CrossAttention(dim)

        self.projection = nn.Linear(dim, 256)

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text features
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state  # (B, T, D)

        # Image features
        image_out = self.image_encoder(
            pixel_values=pixel_values
        ).last_hidden_state  # (B, P, D)

        # Cross attention (text queries image)
        fused = self.cross_attn(text_out, image_out, image_out)

        # Pooling
        fused = fused.mean(dim=1)

        embeddings = self.projection(fused)
        embeddings = F.normalize(embeddings, dim=-1)

        return embeddings

In [7]:
def info_nce_loss(embeddings, temperature=0.07):
    similarity = torch.matmul(embeddings, embeddings.T)

    labels = torch.arange(embeddings.size(0)).to(embeddings.device)

    similarity = similarity / temperature

    loss = F.cross_entropy(similarity, labels)
    return loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = MultiModalModel().to(device)

In [9]:
def train(model, batch_size=32, epochs=5, lr=3e-5):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)

            embeddings = model(input_ids, attention_mask, pixel_values)
            loss = info_nce_loss(embeddings)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pixel_values = batch["pixel_values"].to(device)

                embeddings = model(input_ids, attention_mask, pixel_values)
                loss = info_nce_loss(embeddings)
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

In [13]:
train(model, batch_size=32, epochs=1, lr=3e-5)

Epoch 1/1 [Val]: 100%|██████████| 159/159 [00:39<00:00,  4.04it/s]

Epoch 1/1 — Train Loss: 0.0007, Val Loss: 1.5361


In [14]:
def compute_similarity(model, loader):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)

            emb = model(input_ids, attention_mask, pixel_values)
            all_embeddings.append(emb)

    return torch.cat(all_embeddings)

In [15]:
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [16]:
embeddings = compute_similarity(model, test_loader)

sim_matrix = embeddings @ embeddings.T

def recall_at_k(sim_matrix, k=5):
    correct = 0
    for i in range(sim_matrix.size(0)):
        topk = sim_matrix[i].topk(k).indices
        if i in topk:
            correct += 1
    return correct / sim_matrix.size(0)

print("Recall@1:", recall_at_k(sim_matrix, 1))
print("Recall@5:", recall_at_k(sim_matrix, 5))

100%|██████████| 157/157 [00:28<00:00,  5.48it/s]


Recall@1: 0.2276
Recall@5: 1.0
